# Create global plots of 11-year composites 

### This notebook aimss to recreate the QGIS composite maps I had already made for the first draft. Hoping this makes it a bit more reproducable.

**TODO**: make titles and labels and filenames automatically add number of years and WY START / WY END, etc 

- **Figure 1** 11-yr median runoff onset (arctic stereo)
- **Figure 2** 11-yr median runoff onset, MAD, temporal resolution (arctic stereo)
- **Figure 3** 11-yr median onset, MAD, temporal resolution (robinson)
- **Figure 4** 11-yr annual runoff onset count (robinson)

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import xyzservices as xyz
import rioxarray as rxr
import cartopy.crs as ccrs
from cartopy import feature as cfeature
from global_snowmelt_runoff_onset.config import Config, Tile
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import geopandas as gpd
import numpy as np
import zarr
from matplotlib.colors import BoundaryNorm, ListedColormap
import matplotlib.patheffects as path_effects
from dask.distributed import Client, LocalCluster
from matplotlib.patches import FancyBboxPatch
from matplotlib.patheffects import withStroke, withSimplePatchShadow
from global_snowmelt_runoff_onset.plot_utils import (
    create_month_colorbar,
    create_diverging_colorbar,
)

In [ ]:
config = Config('config/global_config_v10.txt')

In [ ]:
global_hillshade_robinson_da = rxr.open_rasterio('../data/global_hillshade_robinson.tif', masked=True, chunks='auto').squeeze().coarsen(x=10, y=10, boundary='trim').mean().compute()
global_hillshade_robinson_da

In [ ]:
# Read from the v10.0 multiscale pyramid (visualize/pyramid/) instead of the
# retired v9 coarsened store. Level 7 is ~80 m * 2**7 ~= 10.2 km -- comparable
# to the old display grid (factor-20 store coarsened another 5x, ~8 km).
import sys
sys.path.insert(0, '../pyramid')
from build_pyramid import open_pyramid_level
import os

# figures are versioned like results/: v9 renders stay in figures/v9/
FIG_DIR = f'figures/{config.version}'
os.makedirs(FIG_DIR, exist_ok=True)

PYRAMID_LEVEL = 7
global_coarsened_ds = open_pyramid_level(config, PYRAMID_LEVEL)
global_coarsened_ds = global_coarsened_ds.rio.write_crs('EPSG:4326')
global_coarsened_ds

In [ ]:
# Level 7 is already display resolution, so no further coarsening -- just
# subset to the composites and load them (1600 x 3906 x 3 vars, ~75 MB).
composite_vars = ["runoff_onset_median", "runoff_onset_mad", "temporal_resolution_median"]

disp_ds = (
    global_coarsened_ds[composite_vars]
    .astype("float32")
    .rio.write_crs("EPSG:4326")
    .compute()
)
disp_ds

In [ ]:
# Target cartopy projections (numerically equivalent to the EPSG / ESRI codes)
POLAR = ccrs.NorthPolarStereo(true_scale_latitude=71)  # EPSG:3995
ROBIN = ccrs.Robinson()                                # ESRI:54030
PC = ccrs.PlateCarree()

# The data is plotted in its native EPSG:4326 grid with `.plot()` (pcolormesh)
# and transform=PC, letting cartopy do the projection. This is deliberate:
# pre-reprojecting the lon/lat grid with rio.reproject leaves a NaN seam along
# the antimeridian (a visible gap through eastern Russia), whereas cartopy's
# pcolormesh handles the +/-180 wrap correctly. For the Arctic maps we first
# subset to the northern hemisphere so cartopy isn't transforming southern
# cells off toward infinity in polar view.
disp_north = disp_ds.sel(latitude=slice(disp_ds.latitude.max(), 12))

# Polar hillshade: clip the northern part of the Robinson hillshade (so the
# southern hemisphere doesn't reproject off toward infinity) and reproject to
# EPSG:3995. An explicit resolution keeps rioxarray from choosing a tiny pixel
# size. The global Robinson hillshade is reused directly for Figure 3.
global_hillshade_polar_da = (
    global_hillshade_robinson_da.sel(
        y=slice(global_hillshade_robinson_da.y.max(), 0.5e6)
    )
    .rio.reproject("EPSG:3995", resolution=10_000)
)

In [ ]:
# ── shared styling ──────────────────────────────────────────────────────────
MEDIAN = dict(cmap="viridis", vmin=110, vmax=270)  # runoff_onset_median
MAD    = dict(cmap="Reds",    vmin=0,   vmax=30)    # runoff_onset_mad
TRES   = dict(cmap="YlGn_r",  vmin=2,   vmax=14)    # temporal_resolution_median

HS_KW   = dict(cmap="gray", add_colorbar=False, zorder=0, rasterized=True)
DATA_KW = dict(add_colorbar=False, zorder=1, rasterized=True)  # + transform=PC
POLAR_EXTENT = [-7.9e6, 7.9e6, -7.9e6, 7.9e6]  # metres, EPSG:3995 (~25 deg N)


def snug_box(fig, cax, padx=0.012, pady=0.010, rounding=0.04, lw=1.2):
    """Draw a snug white rounded box behind a colorbar axes (incl. its labels).

    The box is auto-sized from the colorbar's tight bounding box (which includes
    the tick labels, axis label and any text drawn on the bar), so it always
    wraps the whole colorbar cleanly regardless of figure size.
    """
    fig.canvas.draw()
    r = fig.canvas.get_renderer()
    bb = cax.get_tightbbox(r).transformed(fig.transFigure.inverted())
    rect = [bb.x0 - padx, bb.y0 - pady, bb.width + 2 * padx, bb.height + 2 * pady]
    p = fig.add_axes(rect)
    p.set_axis_off()
    p.patch.set_visible(False)   # only the FancyBboxPatch fills the box
    p.set_zorder(2)              # above the maps (0/1), below the colorbar (3)
    # mutation_aspect makes the corner radius physically equal in x and y
    # (otherwise a wide, short box gets elliptical, irregular corners)
    figw, figh = fig.get_size_inches()
    aspect = (rect[2] * figw) / (rect[3] * figh)
    p.add_patch(FancyBboxPatch(
        (0, 0), 1, 1, boxstyle=f"round,pad=0,rounding_size={rounding}",
        mutation_aspect=aspect,
        transform=p.transAxes, facecolor="white", edgecolor="black",
        linewidth=lw, clip_on=False))
    cax.set_zorder(3)


def dress_polar(ax, label_size=9, label_lon=-35):
    """Extent + light graticule with white shadowed latitude labels.

    No map frame, no coastlines. Latitude labels are placed manually along the
    `label_lon` meridian (default 45W) and rotated to run along their parallel,
    white with a slight black drop shadow and nudged up off the gridline
    (va='bottom' + 0.2 deg).
    """
    ax.set_extent(POLAR_EXTENT, crs=POLAR)
    ax.spines["geo"].set_visible(False)   # no black frame around the map
    gl = ax.gridlines(
        y_inline=False, x_inline=False, draw_labels=False,
        ylocs=[30, 40, 50, 60, 70, 80], xlocs=range(-180, 181, 60),
        linestyle="-", linewidth=0.5, color="white", alpha=0.35, zorder=4)
    for lat in [30, 40, 50, 60, 70, 80]:
        # rotate the label to follow its parallel: take the local direction of
        # the latitude circle at label_lon in projected coordinates
        x0, y0 = POLAR.transform_point(label_lon - 1, lat, PC)
        x1, y1 = POLAR.transform_point(label_lon + 1, lat, PC)
        rot = np.degrees(np.arctan2(y1 - y0, x1 - x0))
        if abs(rot) > 90:
            rot -= 180 * np.sign(rot)  # keep the text upright
        t = ax.text(label_lon, lat + 0.2, f"{lat}°N", transform=PC,
                    ha="center", va="bottom", rotation=rot,
                    rotation_mode="anchor", color="white",
                    fontsize=label_size, zorder=20)
        t.set_path_effects([withSimplePatchShadow(
            offset=(0.75, -0.75), shadow_rgbFace="black", alpha=1.0)])
    return gl

### Figure 1 - 11-yr median runoff onset (arctic stereo)

In [ ]:
fig = plt.figure(figsize=(10, 10), dpi=100)  # display dpi low; savefig at 300
ax = fig.add_axes([0.03, 0.03, 0.94, 0.90], projection=POLAR)

global_hillshade_polar_da.plot.imshow(ax=ax, transform=POLAR, **HS_KW)
disp_north["runoff_onset_median"].plot(ax=ax, transform=PC, **DATA_KW, **MEDIAN)

dress_polar(ax)
ax.set_title("11-year Median Snowmelt Runoff Onset (WY2015 to WY2025)",
             fontsize=19, weight="bold", pad=8)

# month colorbar (northern-hemisphere months), wide and centred at the top
# of the map, in a snug white box (matching the QGIS layout)
cax = fig.add_axes([0.13, 0.865, 0.74, 0.042])
create_month_colorbar(110, 270, hemisphere="northern", major_tick_spacing=40,
                      cmap="viridis", ax=cax, month_fontsize=20,
                      label_fontsize=17, tick_labelsize=17)
snug_box(fig, cax, padx=0.018, pady=0.014)

fig.savefig(f"{FIG_DIR}/global_composite_median_polar.png",
            bbox_inches="tight", dpi=350)

### Figure 2 - 11-yr median runoff onset, 11-yr MAD, 11-yr temporal resolution (arctic stereo)

In [ ]:
fig = plt.figure(figsize=(15, 15), dpi=100)  # display dpi low; savefig at 300
axa = fig.add_axes([0.075, 0.520, 0.42, 0.42], projection=POLAR)
axb = fig.add_axes([0.505, 0.520, 0.42, 0.42], projection=POLAR)
axc = fig.add_axes([0.290, 0.080, 0.42, 0.42], projection=POLAR)

for ax, var, kw in zip([axa, axb, axc], composite_vars, [MEDIAN, MAD, TRES]):
    global_hillshade_polar_da.plot.imshow(ax=ax, transform=POLAR, **HS_KW)
    disp_north[var].plot(ax=ax, transform=PC, **DATA_KW, **kw)
    dress_polar(ax)
    ax.set_title("")

for ax, letter in zip([axa, axb, axc], ["(a)", "(b)", "(c)"]):
    t = ax.text(0.09, 0.09, letter, transform=ax.transAxes, fontsize=26,
                weight="bold", color="white", zorder=6)
    t.set_path_effects([withStroke(linewidth=3, foreground="black")])

# colorbars centred over their panel, slightly overlapping the map tops
# (a) month colorbar above the top-left map
cax_a = fig.add_axes([0.085, 0.925, 0.40, 0.022])
create_month_colorbar(110, 270, hemisphere="northern", major_tick_spacing=40,
                      cmap="viridis", ax=cax_a, month_fontsize=18,
                      label_fontsize=16, tick_labelsize=16)
snug_box(fig, cax_a, padx=0.010, pady=0.012)

# (b) MAD colorbar above the top-right map
cax_b = fig.add_axes([0.515, 0.925, 0.40, 0.022])
create_diverging_colorbar(
    0, 30, cmap="Reds", ticks=[0, 5, 10, 15, 20, 25, 30], minor_tick_spacing=5,
    left_text="lower variability", right_text="higher variability",
    label="11-year median absolute deviation [days]",
    ax=cax_b, label_fontsize=16, tick_labelsize=16, text_fontsize=18)
snug_box(fig, cax_b, padx=0.010, pady=0.012)

# (c) temporal-resolution colorbar in the gap between the rows
cax_c = fig.add_axes([0.30, 0.482, 0.40, 0.022])
create_diverging_colorbar(
    2, 14, cmap="YlGn_r", ticks=[2, 4, 6, 8, 10, 12, 14], minor_tick_spacing=2,
    left_text="more frequent revisit", right_text="less frequent revisit",
    label="11-year local median temporal resolution [days]",
    ax=cax_c, label_fontsize=16, tick_labelsize=16, text_fontsize=15)
snug_box(fig, cax_c, padx=0.010, pady=0.012)

fig.savefig(f"{FIG_DIR}/global_all_composites_polar.png",
            bbox_inches="tight", dpi=350)

### Figure 3 - 11-yr median runoff onset, 11-yr MAD, 11-yr temporal resolution (robinson)

In [ ]:
fig = plt.figure(figsize=(13, 16), dpi=100)  # display dpi low; savefig at 300
PANEL_H = 0.208
ROBIN_ASPECT = 1.9716  # Robinson map width / height
PANEL_W = PANEL_H * 16 * ROBIN_ASPECT / 13
X0 = (1 - PANEL_W) / 2
axs = [fig.add_axes([X0, y, PANEL_W, PANEL_H], projection=ROBIN)
       for y in (0.760, 0.469, 0.205)]

titles = ["11-year Median Snowmelt Runoff Onset (WY2015 to WY2025)",
          "11-year Median Absolute Deviation (WY2015 to WY2025)",
          "11-year Local Median Temporal Resolution (WY2015 to WY2025)"]

for ax, var, kw, letter, title in zip(
        axs, composite_vars, [MEDIAN, MAD, TRES], ["(a)", "(b)", "(c)"], titles):
    global_hillshade_robinson_da.plot.imshow(ax=ax, transform=ROBIN, **HS_KW)
    disp_ds[var].plot(ax=ax, transform=PC, **DATA_KW, **kw)
    ax.set_global()
    ax.spines["geo"].set_visible(False)   # no border around the map
    ax.gridlines(linestyle="-", linewidth=0.4, color="white", alpha=0.30)
    ax.set_title("")  # clear the default 'spatial_ref = 0' centre title
    ax.set_title(title, loc="center", fontsize=12, weight="bold", pad=3)
    ax.annotate(letter, xy=(0, 1), xycoords="axes fraction",
                xytext=(0, -15), textcoords="offset points",
                fontsize=26, weight="bold", ha="left", va="top",
                annotation_clip=False)

fig.canvas.draw()  # finalise positions before reading them with get_position

CB_TOP = 0.024

def _robin_cbar(ax, cbar_w, cbar_h, make):
    """Place a colorbar (via make(cax)) at the bottom-centre of a subplot + snug box."""
    bb = ax.get_position()
    cx = bb.x0 + bb.width / 2
    cax = fig.add_axes([cx - cbar_w / 2, bb.y0 + CB_TOP - cbar_h,
                        cbar_w, cbar_h])
    make(cax)
    snug_box(fig, cax, pady=0.005)


def _make_month_cbar(c):
    create_month_colorbar(
        110, 270, hemisphere="both", abbreviate_month_names=True,
        major_tick_spacing=40, cmap="viridis", ax=c, month_fontsize=12,
        label_fontsize=13, tick_labelsize=13,
        label="11-year median runoff onset date [day of water year]")
    # hemisphere key, styled to mirror the month labels on the bar itself:
    # NH bold / SH bold-italic in parens, white with a black outline
    # (y positions are in bar-height units, tuned to the 0.026 bar)
    kw = dict(transform=c.transAxes, va="center", color="white",
              weight="bold", clip_on=False)
    t1 = c.text(0.5, -1.72, "N. hemisphere month", ha="center",
                fontsize=12, **kw)
    t2 = c.text(0.5, -2.12, " (S. hemisphere month)", ha="center",
                fontsize=10, style="italic", **kw)
    for t in (t1, t2):
        t.set_path_effects([withStroke(linewidth=2, foreground="black")])


_robin_cbar(axs[0], 0.47, 0.026, _make_month_cbar)
_robin_cbar(axs[1], 0.44, 0.019, lambda c: create_diverging_colorbar(
    0, 30, cmap="Reds", ticks=[0, 5, 10, 15, 20, 25, 30], minor_tick_spacing=5,
    left_text="lower variability", right_text="higher variability",
    label="11-year median absolute deviation [days]", ax=c,
    label_fontsize=13, tick_labelsize=13, text_fontsize=15))
_robin_cbar(axs[2], 0.44, 0.019, lambda c: create_diverging_colorbar(
    2, 14, cmap="YlGn_r", ticks=[2, 4, 6, 8, 10, 12, 14], minor_tick_spacing=2,
    left_text="more frequent revisit", right_text="less frequent revisit",
    label="11-year local median temporal resolution [days]", ax=c,
    label_fontsize=13, tick_labelsize=13, text_fontsize=14))

fig.savefig(f"{FIG_DIR}/global_all_composites_robinson_long.png",
            bbox_inches="tight", dpi=350)

In [ ]:
# new version with bigger map area: full-page layout matching the QGIS draft.
# Antarctica is cropped away (no data there), the maps span nearly the full
# figure width, and the shrunken colorbars sit inside each map over the
# southern-ocean gap between the Andes and eastern Australia so they don't
# cover any data.
LAT_MIN = -57  # southernmost latitude shown: keeps Patagonia, drops Antarctica

Y_TOP = ROBIN.transform_point(0, 90, PC)[1]
Y_BOT = ROBIN.transform_point(0, LAT_MIN, PC)[1]
X_RIGHT = ROBIN.transform_point(180, 0, PC)[0]
CROP_ASPECT = 2 * X_RIGHT / (Y_TOP - Y_BOT)  # cropped map width / height

FIGW, PANEL_W = 10, 0.96
TITLE_IN = 0.32                              # vertical room per panel title [in]
panel_h_in = PANEL_W * FIGW / CROP_ASPECT    # map panel height [in]
FIGH = 3 * (panel_h_in + TITLE_IN)           # ~13.4 in -> full-page proportions
fig = plt.figure(figsize=(FIGW, FIGH), dpi=100)  # display dpi low; savefig at 350

X0 = (1 - PANEL_W) / 2
SLOT = (panel_h_in + TITLE_IN) / FIGH        # title + map, in figure fraction
PANEL_H = panel_h_in / FIGH
axs = [fig.add_axes([X0, 1 - (i + 1) * SLOT, PANEL_W, PANEL_H], projection=ROBIN)
       for i in range(3)]

titles = ["11-year Median Snowmelt Runoff Onset (WY2015 to WY2025)",
          "11-year Median Absolute Deviation (WY2015 to WY2025)",
          "11-year Local Median Temporal Resolution (WY2015 to WY2025)"]

for ax, var, kw, letter, title in zip(
        axs, composite_vars, [MEDIAN, MAD, TRES], ["(a)", "(b)", "(c)"], titles):
    global_hillshade_robinson_da.plot.imshow(ax=ax, transform=ROBIN, **HS_KW)
    disp_ds[var].plot(ax=ax, transform=PC, **DATA_KW, **kw)
    ax.set_global()
    ax.set_ylim(Y_BOT, Y_TOP)   # crop Antarctica for a bigger map area
    ax.spines["geo"].set_visible(False)   # no border around the map
    ax.gridlines(linestyle="-", linewidth=0.4, color="white", alpha=0.30)
    ax.set_title("")  # clear the default 'spatial_ref = 0' centre title
    ax.set_title(title, loc="center", fontsize=15, weight="bold", pad=3)
    ax.annotate(letter, xy=(0, 1), xycoords="axes fraction",
                xytext=(0, -15), textcoords="offset points",
                fontsize=26, weight="bold", ha="left", va="top",
                annotation_clip=False)

fig.canvas.draw()  # finalise positions before reading them with get_position

# colorbars sit inside each map, in the ocean gap between the Andes and
# eastern Australia -- CB_XFRAC/y_frac position their centre within the panel
CB_XFRAC = 0.595


def _robin_cbar_inside(ax, cbar_w, cbar_h, y_frac, make):
    """Colorbar (via make(cax)) inside a panel: centred at CB_XFRAC of its
    width, y_frac up from its bottom."""
    bb = ax.get_position()
    cax = fig.add_axes([bb.x0 + CB_XFRAC * bb.width - cbar_w / 2,
                        bb.y0 + y_frac * bb.height, cbar_w, cbar_h])
    make(cax)
    snug_box(fig, cax, padx=0.008, pady=0.004)


def _make_month_cbar_small(c):
    create_month_colorbar(
        110, 270, hemisphere="both", abbreviate_month_names=True,
        major_tick_spacing=40, cmap="viridis", ax=c, month_fontsize=10,
        label_fontsize=10, tick_labelsize=10,
        label="11-year median runoff onset date [day of water year]")
    # hemisphere key below the bar (y positions in bar-height units,
    # tuned to the 0.020 bar so it clears the axis label above it)
    kw = dict(transform=c.transAxes, va="center", color="white",
              weight="bold", clip_on=False)
    t1 = c.text(0.5, -2.40, "N. hemisphere month", ha="center",
                fontsize=10, **kw)
    t2 = c.text(0.5, -3.00, " (S. hemisphere month)", ha="center",
                fontsize=9, style="italic", **kw)
    for t in (t1, t2):
        t.set_path_effects([withStroke(linewidth=2, foreground="black")])


_robin_cbar_inside(axs[0], 0.50, 0.022, 0.32, _make_month_cbar_small)
_robin_cbar_inside(axs[1], 0.50, 0.025, 0.32, lambda c: create_diverging_colorbar(
    0, 30, cmap="Reds", ticks=[0, 5, 10, 15, 20, 25, 30], minor_tick_spacing=5,
    left_text="lower variability", right_text="higher variability",
    label="11-year median absolute deviation [days]", ax=c,
    label_fontsize=10, tick_labelsize=10, text_fontsize=11))
_robin_cbar_inside(axs[2], 0.50, 0.025, 0.32, lambda c: create_diverging_colorbar(
    2, 14, cmap="YlGn_r", ticks=[2, 4, 6, 8, 10, 12, 14], minor_tick_spacing=2,
    left_text="more frequent revisit", right_text="less frequent revisit",
    label="11-year local median temporal resolution [days]", ax=c,
    label_fontsize=10, tick_labelsize=10, text_fontsize=10))

fig.savefig(f"{FIG_DIR}/global_all_composites_robinson_wide.png",
            bbox_inches="tight", dpi=350)

### Figure 4 - 11-yr annual runoff onset count (robinson)

In [ ]:
# Count of water years with data per level-7 (~10.2 km) cell. A pyramid cell
# is non-NaN iff ANY native pixel beneath it had data that year (mean-of-valid
# propagates any-valid upward), matching the old max-then-notnull semantics.
runoff_onset_count_da = (
    global_coarsened_ds["runoff_onset"]
    .notnull()
    .sum(dim="water_year", dtype="int16")
    .compute()
)

In [ ]:
ROBIN_ASPECT = 1.9716  # Robinson map width / height
FIG_W, FIG_H = 13, 7.5

fig = plt.figure(figsize=(FIG_W, FIG_H), dpi=100)  # display dpi low; savefig at 300

PANEL_H = 0.86
PANEL_W = PANEL_H * FIG_H * ROBIN_ASPECT / FIG_W
X0 = (1 - PANEL_W) / 2
Y0 = 0.03
ax = fig.add_axes([X0, Y0, PANEL_W, PANEL_H], projection=ROBIN)

global_hillshade_robinson_da.plot.imshow(ax=ax, transform=ROBIN, **HS_KW)

# discrete bins: one color per integer count 1-10
vmin, vmax = 1, 11
n_bins = vmax - vmin + 1
cmap = ListedColormap(plt.cm.Oranges(np.linspace(0.15, 1.0, n_bins)))
bounds = np.arange(vmin - 0.5, vmax + 1.5, 1)
norm = BoundaryNorm(bounds, cmap.N)

runoff_onset_count_da.where(runoff_onset_count_da > 0).plot(
    ax=ax, transform=PC, cmap=cmap, norm=norm, zorder=1,
    rasterized=True, add_colorbar=False)

ax.set_global()
ax.spines["geo"].set_visible(False)
ax.gridlines(linestyle="-", linewidth=0.4, color="white", alpha=0.30)
ax.set_title("")
ax.set_title("11-year Annual Runoff Onset Count (WY2015 to WY2025)", loc="center",
              fontsize=15, weight="bold", pad=6)

fig.canvas.draw()

# ── inset, discrete, horizontal colorbar (mirrors _robin_cbar in the triptych) ──
CB_TOP = 0.08
cbar_w, cbar_h = 0.6, 0.08

bb = ax.get_position()
cx = bb.x0 + bb.width / 2
cax = fig.add_axes([cx - cbar_w / 2, bb.y0 + CB_TOP - cbar_h, cbar_w, cbar_h])

cb = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), cax=cax,
                   orientation="horizontal", boundaries=bounds,
                   ticks=np.arange(vmin, vmax + 1), spacing="proportional",
                   drawedges=True)

cb.outline.set_linewidth(0.6)
cb.dividers.set_color("white")
cb.dividers.set_linewidth(0.8)

cb.set_ticks([]) 

for i in range(len(bounds) - 1):
    # Calculate the horizontal center of the current patch
    patch_center = (bounds[i] + bounds[i+1]) / 2
    
    # Calculate the actual integer value represented by this patch
    patch_value = int(vmin + i)
    
    # Add text. We use the colorbar's data coordinates for X, and center it vertically (Y=0.5)
    txt = cax.text(
        x=patch_center,
        y=0.5,
        s=str(patch_value),
        color="white",       # Core text color
        fontsize=16,
        weight="bold",
        va="center",
        ha="center",
        transform=cax.transData
    )

    txt.set_path_effects([
        path_effects.withStroke(linewidth=2.5, foreground="black")
    ])

cb.set_label("number of water years", fontsize=13)

snug_box(fig, cax, pady=0.02)  # same helper used in the triptych

fig.savefig(f"{FIG_DIR}/global_allyrs_annual_runoff_onset_count.png",
            bbox_inches="tight", dpi=350)

: 